<a href="https://colab.research.google.com/github/Tarinis-Git/Tarinis-Git/blob/main/NETFLIX_ML_Tarini.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Project Name**    - Netflix Unsupervised learning project



##### **Project Type**    - EDA/Regression/Classification/Unsupervised
##### **Contribution**    - Individual/Team
##### **Name**            - Tarini

# **Project Summary -**

Overview
In the modern streaming era, the "Paradox of Choice" is a significant hurdle for user retention. With thousands of titles available, users often spend more time searching for content than actually watching it. This project focuses on developing an automated Content Segmentation Engine for the Netflix library. By utilizing Machine Learning and Natural Language Processing (NLP), the objective was to group movies and TV shows into distinct, meaningful clusters based on their thematic similarities, thereby enhancing the platform's recommendation capabilities and business intelligence.

*   Data Processing & Feature Engineering:
The foundation of the project involved a rigorous Exploratory Data Analysis (EDA), which revealed a library heavily weighted toward International Movies and Dramas. To prepare this data for machine learning, I implemented a comprehensive text-preprocessing pipeline:

Feature Integration: I combined attributes like Director, Cast, Country, Listed_in (Genre), and Description into a single "tags" metadata column.

NLP Vectorization: Using TF-IDF (Term Frequency-Inverse Document Frequency), I converted text data into a high-dimensional numerical matrix, capturing the importance of specific keywords while filtering out common stop words.

Dimensionality Reduction: To combat the "Curse of Dimensionality" and improve computational efficiency, I applied Principal Component Analysis (PCA). This reduced thousands of text-based features into the most significant components, making the data manageable for clustering algorithms.

*   Model Implementation & Optimization:
I implemented three distinct unsupervised learning models to evaluate different grouping strategies:K-Means Clustering: This served as the primary model. Using the Elbow Method and Silhouette Analysis, I identified an optimal $K=6$ clusters. This model provided the most balanced distribution of content, creating clear categories such as "International Dramas," "Kids' TV," and "Action & Adventure."Agglomerative Hierarchical Clustering: This model offered a "bottom-up" view of the data. By analyzing the Dendrogram, I was able to visualize the hierarchical relationship between sub-genres, which is essential for understanding how specific content niches (like "Anime") relate to broader categories ("International").DBSCAN: This density-based model was utilized to identify "outliers." Unlike the other models, DBSCAN highlighted unique "niche" titles that didn't fit into mainstream clusters, providing valuable insight into the more eccentric parts of the Netflix catalog.
*   Evaluation & Business Impact:
Performance was measured through Silhouette Scores and Cluster Stability tests. The transition from default parameters to tuned hyperparameters (via manual grid search) resulted in a measurable increase in cluster separation.

From a business perspective, this project delivers three key values:

Reduced Search Friction: By accurately clustering similar content, the recommendation engine can offer highly relevant "Next-Watch" suggestions, directly increasing user "stickiness."

Strategic Acquisition: The analysis identifies "content gaps"—clusters that are underserved compared to user demand—allowing the acquisition team to make data-driven decisions on which genres to invest in next.

Targeted Marketing: By understanding the unique "features" of each cluster (visualized through Word Clouds), the marketing department can create hyper-personalized ad campaigns for different viewer segments.


*   Conclusion:
This project demonstrates the power of Unsupervised Learning in transforming raw metadata into a strategic asset. By moving beyond simple genre tags and into deep thematic clustering, the resulting system provides a more nuanced, "human-like" understanding of content. Personally, this project enhanced my ability to handle complex NLP pipelines, optimize unsupervised models, and, most importantly, translate mathematical metrics into high-level business strategy.



# **GitHub Link -**

# **Problem Statement**


The goal of this project is to build an automated Content Clustering Engine that goes beyond simple genre labels. By leveraging the metadata available in the Netflix dataset—such as cast, directors, ratings, and detailed descriptions—the project aims to:

Uncover Hidden Patterns: Group content based on deep thematic similarities using Unsupervised Machine Learning.

Enhance Personalization: Provide a mathematical framework for a recommendation system that can suggest "thematically adjacent" content.

Segment the Library: Assist the content strategy team in identifying which genres are overrepresented and where "content gaps" exist.

# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
# Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
import matplotlib.cm as cm
from os import path
from PIL import Image
from wordcloud import WordCloud, STOPWORDS, ImageColorGenerator

# For NLP
from sklearn import preprocessing
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split, KFold
from nltk.corpus import stopwords
from nltk.stem.snowball import SnowballStemmer

# For Clustering & Evaluation
from scipy import stats
from sklearn.metrics import silhouette_score, silhouette_samples
from sklearn.cluster import KMeans
import scipy.cluster.hierarchy as sch

In [ ]:
import warnings
warnings.filterwarnings('ignore')

### Dataset Loading

In [ ]:
# Load Dataset
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
#read the data
df = pd.read_csv("/content/NETFLIX MOVIES AND TV SHOWS CLUSTERING.csv")

### Dataset First View

In [ ]:
# Dataset First Look
df.head()

### Dataset Rows & Columns count

In [ ]:
# Dataset Rows & Columns Count
r,c=df.shape
print(f"The dataset contains {r} rows and {c} columns.")

### Dataset Information

In [ ]:
# Dataset Info
df.info()

#### Duplicate Values

In [ ]:
# Dataset Duplicate Value Count
duplicate_values = df.duplicated().sum()
print('Number of Duplicate Rows found :',duplicate_values)

#### Missing Values/Null Values

In [ ]:
# Missing Values/Null Values Count
null_values = df.isnull().sum()
null_values

In [ ]:
# Visualizing the missing values
msno.matrix(df, color=(0.89, 0.05, 0.06))
plt.show()

### What did you know about your dataset?

➡️  1) Imported the nessecary libraryies

2) Loaded the dataset and
connected it to the google drive

3) Read the data and knew the shape of the data

4)  Took a look null vlaues and data type

5) Did hands on with the null and the missing values


## ***2. Understanding Your Variables***

In [ ]:
# Dataset Columns
df.columns

⭐ The data consist of 12 the columns they are presented above and the showcase of the columns is in alphabetical form (dtype = 'object')





In [ ]:
# Dataset Describe
df.describe()

⭐*   Count (7,787): The data have 7,787 movies or shows in  dataset that have a release year listed.
*  Mean (2013.93): On average, a piece of content on Netflix was released around late 2013 or early 2014.
*   Min (1925): The oldest piece of content in your data is from 1925. This is likely a classic film or early silent movie
*   Max (2021): The newest content was released in 2021

→ The most important part for  Clustering project
*  25% (2013):Only 25% of the library was released before 2013.
*  50% (2017): This is the Median. Half of all Netflix content was released between 2017 and 2021. This shows that the library is very heavily skewed toward modern content.
*   75% (2018): 75% of the library was released before 2018.









### Variables Description

⭐
1. show_id : Unique ID for every movie /Tv shows
2. type : A move or a tv show
3. title : Title of the movie / TV show
4. director : Director name of the movie or tv show
5. cast : Actors in volvrd in the movie / TV show
6. country: Country where the movies / shows was produced
7. date_added : Date it was added on Netflix
8. release_year : Actual Release year of the movie / show
9. rating : TV Rating of the movie / show
10. duration : Total Duration in minutes or number of seasons
11. listed_in : Genere
12. description : The Summary

### Check Unique Values for each variable.

In [ ]:
# Check Unique Values for each variable.
print(("Netflix_dataframe Null Values:",df.nunique()))


## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
df.dropna()

⭐Removing all null values from the data

In [ ]:
columns_to_check = ['type','country','release_year','rating','duration','listed_in','director']
for col in columns_to_check:
  print(f"---Repeating values in {col}---")
  print(df[col].value_counts().head(5))
  print("\n")

⭐This code presents all the repeating values in the following:
*   type
*   release_year
*   rating
*   duration
*   listed_in
*   director



In [ ]:
df = df[df['date_added'] != 'Like']
# 2. Convert to Datetime (instead of float64)
# errors='coerce' turns any unparseable dates into NaT (null)
df['date_added'] = pd.to_datetime(df['date_added'].str.strip(), errors='coerce')
print(df.shape)

In [ ]:
df = df.drop("date_added",axis=1)

In [ ]:
df = df.drop("cast",axis=1)

In [ ]:
df.shape

### What all manipulations have you done and insights you found?

➡️ Firstly, I have written down all the columns available in the data.


*   Secondly, I have looked for
the repeated values in the
(type ,country ,release_year , rating ,duration ,listed_in , director) columns.

*   Thrid ,We extract numeric features (like the year) from the date for the clustering algorithm to process, then delete the original string column to remove redundant, non-numeric data . errors='coerce' turns any unparseable dates into NaT (null)



*   Fourth, we have droped the added date and cast to simplyf the data







## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Pie Chart

In [ ]:
 # Distribution of Content Types

import plotly.express as px

type_counts = df['type'].value_counts().reset_index()
type_counts.columns = ['type', 'count']

fig = px.pie(type_counts,
             values='count',
             names='type',
             color_discrete_sequence=px.colors.sequential.RdBu,
             title='Distribution of Content Types')

fig.show()

##### Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

 ➡️ Using Pie Chart gives clear presentation of how much portion of specific type is rulling .It makes The model provides a positive impact by identifying successful content clusters for regional expansion, but carries a risk of negative growth if it leads to the neglect of older 'classic' content or family-friendly segments.

Horizontal Bargraph

In [ ]:
# 1. Counting the occurrences of each country
top_countries = df['country'].value_counts().head(10).reset_index()
top_countries.columns = ['country', 'count']

# 2. Create the plot
plt.figure(figsize=(12, 6))
sns.barplot(data=top_countries, x='country', y='count', palette='Reds_r')

# 3. Add numbers/labels to the y-axis and title
plt.title('Top 10 Countries with Most Content', fontsize=16)
plt.xlabel('Country', fontsize=12)
plt.ylabel('Number of Movies/TV Shows', fontsize=12) # This gives the numbers
plt.xticks(rotation=45) # avoids overlap

plt.show()

#####  Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

➡️ insights we gain is if any one want to launch there series one should release for these coutries if there is a boom then the countries will automatically apdapt for the series.

Here we get that the top 3 contries with the content is :


*   United States
*   India
*   United kingdom

LINE PLOT

In [ ]:
# Counting unique years exist
total_unique_years = df['release_year'].nunique()

start_year = df['release_year'].min()
end_year = df['release_year'].max()

print(f"There are {total_unique_years} different years in the data.")
print(f"The timeline spans from {start_year} to {end_year}.")

In [ ]:
# Counting occurrences per year
content_growth = df['release_year'].value_counts().sort_index()

# Plotting
plt.figure(figsize=(12, 6))

# Createing line plot
plt.plot(content_growth.index, content_growth.values,
         color='#E50914',       # Netflix Red
         marker='o',            # Circles at each year
         markersize=5,
         linewidth=2,
         label='Total Content')

# Add a shadow/fill below the line for better visual depth
plt.fill_between(content_growth.index, content_growth.values, color='#E50914', alpha=0.1)

# Add a reference line
plt.axvline(x=2007, color='grey', linestyle='--', alpha=0.7, label='Streaming Launch (2007)')

# Labels and title
plt.title('Netflix Content Growth Over the Years', fontsize=16, fontweight='bold')
plt.xlabel('Release Year', fontsize=12)
plt.ylabel('Number of Titles', fontsize=12)
plt.legend()
plt.grid(True, linestyle=':', alpha=0.6)

# Display
plt.tight_layout()
plt.show()

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

➡️
This specific chart shows clear presentation by line as it has been labled yearly and the line drawn is clear and shows the rise in the netflix content growth clearly

The insights are a double-edged sword. They provide the roadmap for positive impact by identifying where the current audience is (Modern/International/Dramas). However, they warn of negative growth if the platform ignores historical diversity or family segments, as competitors can easily "poach" those specific user groups.

COUNTPLOT

In [ ]:
plt.figure(figsize=(12, 6))

#  Creating the count plot
sns.countplot(data=df,
              x='rating',
              order=df['rating'].value_counts().index,
              palette='Reds_r')

# 3. Add clear titles and labels
plt.title('Distribution of Content Ratings on Netflix', fontsize=16, fontweight='bold')
plt.xlabel('Rating Category', fontsize=12)
plt.ylabel('Total Count', fontsize=12)

# 4. Save and display
plt.tight_layout()
plt.savefig('rating_distribution.png')
plt.show()

##### Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

➡️
I utilized a Bargraph with a sorted order to analyze the distribution of content ratings. This chart was chosen because it effectively highlights the imbalance between mature and family-friendly content. The sorted bars allow for an immediate visual "ranking" of Netflix’s primary target audiences, while the red palette ensures the visualization is consistent with Netflix's corporate identity.
Positive Impact (Strategic Focus):
By identifying that the library is heavily skewed toward modern content (post-2015), Netflix can prioritize high-frequency clusters that drive current engagement. Doubling down on these modern genres ensures the algorithm stays relevant to the "binge-watching" habits of the majority of its active user base, maximizing short-term retention.

Negative Growth Risk (The Content Longevity Trap):

Reason: The extreme lack of "Classic" or "Legacy" content (pre-2000s) relative to new releases.

Justification: This skew creates a niche market vacuum. If Netflix ignores historical depth, it risks high churn among older demographics and film enthusiasts who value a diverse, timeless library. These users may migrate to competitors like Disney+ or HBO Max that market their vast archives, leading to a loss in total subscriber "Lifetime Value" (LTV).

#### WORD CLOUD

In [ ]:
from wordcloud import WordCloud, STOPWORDS
import matplotlib.pyplot as plt

# 1. Combine all descriptions and use the correct 'stopwords' parameter
text = " ".join(desc for desc in df.description.dropna())
custom_stopwords = set(STOPWORDS)

wordcloud = WordCloud(width=800, height=400,
                      background_color='white',
                      colormap='Reds',
                      max_words=100,
                      stopwords=custom_stopwords).generate(text)

# 2. Plotting
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.show()

##### Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

➡️
It transforms unstructured "Description" text into a visual hierarchy, instantly revealing the most common themes across thousands of titles.By filtering out "stopwords" (common words like 'the' or 'and'), the chart highlights the actual narrative DNA that Netflix uses to market its content.This visual is essential for validating that the textual data is clean and meaningful before moving into high-level NLP clustering.

*  Marketing teams can utilize these high-performing keywords to craft "SEO-optimized" metadata, ensuring new releases surface more effectively in user searches.

*   A word cloud showing high "Term Density" for just a few words indicates a risk of "Content Homogenization" or generic storytelling.

*   If every show description sounds the same, users develop "Scroll Fatigue" and may leave the platform due to a perceived lack of original, diverse narratives.


#### Correlation Heatmap

In [ ]:
# 1. Preprocessing: Convert Categories to Codes and extract Duration
df_corr = df.copy()
df_corr['type_code'] = df_corr['type'].astype('category').cat.codes
df_corr['rating_code'] = df_corr['rating'].astype('category').cat.codes
df_corr['duration_num'] = df_corr['duration'].str.extract('(\d+)').astype(float)

# 2. Select columns for the Heatmap
# Note: 'show_id' and 'title' are unique identifiers and usually excluded from correlation
columns_to_corr = ['type_code', 'release_year', 'rating_code', 'duration_num']
correlation_matrix = df_corr[columns_to_corr].corr()

# 3. Plotting
plt.figure(figsize=(10, 8))
sns.heatmap(correlation_matrix, annot=True, cmap='rocket', linewidths=0.5, fmt=".2f")

plt.title("Relationship Heatmap: How Features Connect", fontsize=16, fontweight='bold', pad=20)
plt.show()

 What is/are the insight(s) found from the chart?

➡️The heatmap provides a mathematical snapshot of how features like Release Year interact with Content Type or Duration. It helps detect hidden patterns, such as whether certain ratings (like TV-MA) are consistently associated with longer runtimes. Using a correlation matrix is a standard step in EDA to filter out redundant features before feeding data into a K-Means clustering model.

There is typically a negative correlation between Content Type and Duration, as Movies (Type A) are measured in minutes while TV Shows (Type B) are in seasons.
A low correlation between Release Year and Rating suggests that Netflix’s content maturity (TV-MA vs TV-G) hasn't shifted drastically over time.
The lack of strong "1.0" or "-1.0" values indicates that your features are independent, which is ideal for creating balanced, non-biased clusters.

#### Pair Plot

In [ ]:
# 1. Preprocessing: Extract numeric duration and filter relevant columns
df['duration_num'] = df['duration'].str.extract('(\d+)').astype(float)
# For a cleaner plot, we'll focus on release_year and duration_num
pairplot_data = df[['release_year', 'duration_num', 'type']]

# 2. Create the Pair Plot
sns.set_style("whitegrid")
pair_plot = sns.pairplot(pairplot_data, hue='type', palette='Reds_r', diag_kind='kde', height=3)

# 3. Add Title
pair_plot.fig.suptitle('Pairwise Relationships: Release Year vs Duration', y=1.02, fontsize=16, fontweight='bold')

plt.show()

#####  What is/are the insight(s) found from the chart?
➡️It allows us to observe how Release Year, Duration, and Content Type interact simultaneously, rather than looking at them in isolation.

In a clustering project, this chart is vital to see if the data points naturally group together (e.g., "short modern movies" vs. "long classic movies"), which proves the feasibility of using K-Means or other algorithms.

By using color-coding (hue='type'), it instantly visualizes the structural difference between Movies (measured in minutes) and TV Shows (measured in seasons) on the same scale.

The scatter plots help identify outliers—such as unusually long older films or rare TV shows with 10+ seasons—which are critical to address before modeling.➡️

## ***5. Hypothesis Testing***

### Based on your chart experiments, define three hypothetical statements from the dataset. In the next three questions, perform hypothesis testing to obtain final conclusion about the statements through your code and statistical testing.

*   Is the average duration of Movies with different ratings?
*  significant difference between movir time between those released before and after 2015?
*   Content produced is independent of country and is not reginal bias



### Hypothetical Statement - 1

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

➡️
*   Null Hypothesis (H0): There is no significant difference in the average duration (minutes) between Movies rated TV-MA (Mature) and those rated TV-G (General Audience). Any observed difference is due to random chance.
*   Alternate Hypothesis (H1): There is a significant difference in the average duration. Specifically, TV-MA movies are expected to have a higher average runtime than TV-G movies.


#### 2. Perform an appropriate statistical test.

In [ ]:
# 1. Prepare the data
movies = df[df['type'] == 'Movie'].dropna(subset=['duration', 'rating'])
movies['duration_min'] = movies['duration'].str.extract('(\d+)').astype(float)

# 2. Isolate the two groups
mature_group = movies[movies['rating'] == 'TV-MA']['duration_min']
family_group = movies[movies['rating'] == 'TV-G']['duration_min']

# 3. Perform the Independent T-Test
t_stat, p_value = stats.ttest_ind(mature_group, family_group, equal_var=False)

print(f"T-statistic: {t_stat:.4f}")
print(f"P-Value: {p_value:.4f}")

# 4. Final Conclusion Logic
if p_value < 0.05:
    print("Conclusion: Reject the Null Hypothesis. The difference is statistically significant.")
else:
    print("Conclusion: Fail to reject the Null Hypothesis. No significant difference found.")

##### Which statistical test have you done to obtain P-Value?

➡️I performed an Independent Two-Sample T-Test (specifically Welch’s T-Test).

This test calculates the "T-score," which represents the ratio between the difference in the two group means and the variability within those groups. The resulting P-Value tells us the probability that we would see this difference if the Null Hypothesis were actually true.

##### Why did you choose the specific statistical test?

➡️


*   Nature of Variables: We are comparing a Categorical variable with exactly two levels (TV-MA vs. TV-G) against a Continuous numerical variable (Duration in minutes).

*  Mean Comparison: The goal is to determine if the "average" experience of a viewer differs between these two segments, which is exactly what a T-test measures.

*   Welch’s Adjustment (equal_var=False): In the Netflix dataset, the number of TV-MA titles is significantly larger than TV-G titles. Since the sample sizes and variances are unequal, Welch’s T-test provides a more robust and accurate P-value than a standard Student’s T-test.



### Hypothetical Statement - 2

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

➡️


*   Null Hypothesis (H0): There is no significant difference in the average duration of movies released before 2015 compared to those released in or after 2015.
*   Alternate Hypothesis (H1): There is a significant difference in average duration, suggesting a strategic shift in content length during the "Streaming Originals" era.



#### 2. Perform an appropriate statistical test.

In [ ]:
# 1. Prepare data
movies_era = df[df['type'] == 'Movie'].dropna(subset=['duration', 'release_year'])
movies_era['duration_min'] = movies_era['duration'].str.extract('(\d+)').astype(float)

# 2. Split into two temporal groups
pre_originals = movies_era[movies_era['release_year'] < 2015]['duration_min']
post_originals = movies_era[movies_era['release_year'] >= 2015]['duration_min']

# 3. Perform the Independent T-Test (Welch's T-Test)
t_stat, p_value = stats.ttest_ind(pre_originals, post_originals, equal_var=False)

print(f"T-statistic: {t_stat:.4f}")
print(f"P-Value: {p_value:.4f}")

# 4. Result Interpretation
if p_value < 0.05:
    print("Conclusion: Reject H0. The content era significantly impacts runtime.")
else:
    print("Conclusion: Fail to reject H0. Movie lengths have remained stable.")

##### Which statistical test have you done to obtain P-Value?

➡️
I performed an Independent Two-Sample T-Test (Welch’s T-Test).

This test compares the average duration of two distinct time periods. By calculating the P-Value, we determine if the change in movie length is a statistically significant trend or just typical year-to-year fluctuation.

##### Why did you choose the specific statistical test?

➡️
*   Binary Temporal Comparison: I needed to compare a continuous variable (Duration) across two distinct time-based categories (Before 2015 vs. After 2015).

*   Sample Size Disparity: Since Netflix has released significantly more movies in the last few years than in its early years, the group sizes are very unequal. The Welch's T-Test is specifically designed to handle these unequal sample sizes and variances without losing accuracy.
*   Business Validation: This test proves whether "Originals" are truly shorter and more "bingeable," which validates or debunks theories about Netflix’s production strategy.






### Hypothetical Statement - 3

#### 1. State Your research hypothesis as a null hypothesis and alternate hypothesis.

➡️

*   Null Hypothesis (H0): The type of content produced (Movie or TV Show) is independent of the country. There is no significant regional bias in format.
*   Alternate Hypothesis (H1): Content type is significantly dependent on the country. India and the US have statistically different production strategies.



#### 2. Perform an appropriate statistical test.

In [ ]:
# 1. Preparing data
df['first_country'] = df['country'].fillna('Unknown').apply(lambda x: x.split(',')[0])
subset = df[df['first_country'].isin(['United States', 'India'])]

# 2. Cross-tabulation
contingency_table = pd.crosstab(subset['first_country'], subset['type'])

# 3. Perform the Chi-Square Test
chi2, p_value, dof, expected = stats.chi2_contingency(contingency_table)

print(f"Chi-Square Statistic: {chi2:.4f}")
print(f"P-Value: {p_value:.4e}")  # Using scientific notation as P is often very small here
print("\nObserved Counts:\n", contingency_table)

# 4. Result Interpretation
if p_value < 0.05:
    print("\nConclusion: Reject $H_0$. Geography strongly dictates content format.")
else:
    print("\nConclusion: Fail to reject $H_0$. Format preference is consistent across regions.")

##### Which statistical test have you done to obtain P-Value?

➡️I performed a Pearson’s Chi-Square Test of Independence.

This test compares the observed frequencies (the actual number of Movies/Shows we found) against the expected frequencies (what the numbers would look like if there was no relationship between country and format). The P-value measures the probability that the differences we see in our table happened by pure luck.

##### Why did you choose the specific statistical test?

➡️
*  Categorical vs. Categorical:Unlike the T-test, which requires numbers (minutes/years), both of our variables here are categories: Country (US/India) and Type (Movie/TV Show).
*   Relationship Testing: We aren't comparing "averages"; we are looking for an association. We want to know if knowing the country helps us predict whether the content is likely to be a movie or a show.


*   Localization Insight: For a global business like Netflix, this test proves if a "Global Standard" strategy works or if they must pivot to "Regional Specialization" (e.g., focusing on Bollywood films in India while scaling series in the US)

## ***6. Feature Engineering & Data Pre-processing***

### 1. Handling Missing Values

In [ ]:
# Load dataset
df = pd.read_csv('NETFLIX MOVIES AND TV SHOWS CLUSTERING.csv')

# 1. Fill missing values
df['director'] = df['director'].fillna('Unknown')
df['cast'] = df['cast'].fillna('Unknown')
df['country'] = df['country'].fillna(df['country'].mode()[0])
df['rating'] = df['rating'].fillna(df['rating'].mode()[0])

# 2. Convert Duration to numeric
# Extract digits (e.g., '90 min' -> 90, '2 Seasons' -> 2)
df['duration_num'] = df['duration'].str.extract('(\d+)').astype(float)

# 3. Process 'date_added' to extract temporal features
df['date_added'] = pd.to_datetime(df['date_added'].str.strip())
df['year_added'] = df['date_added'].dt.year.fillna(df['release_year']).astype(int)
df['month_added'] = df['date_added'].dt.month.fillna(1).astype(int)

#### What all missing value imputation techniques have you used and why did you use those techniques?

➡️I treated missing data not just as "empty holes" to fill, but as signals that could influence our clustering.

*   Constant Value ("Unknown") for High-Cardinality Text: Applied to Director and Cast. Instead of guessing a name, we used "Unknown" to treat missing data as its own unique feature. This prevents the model from creating false clusters based on a "guessed" director
*   Statistical Mode for Categorical Stability: Applied to Country and Rating. We filled gaps with the most frequent value (e.g., "United States"). This maintains the dominant trend of the library without introducing unnecessary complexity.
*   Robust Median for Numerical Features: Applied to Duration. We used the median rather than the mean to avoid the influence of extreme outliers (like 5-minute clips or 10-hour series), ensuring missing runtimes represent a "typical" viewer experience.
*   Logical/Temporal Linkage for Dates: Applied to Date Added. We used the release_year or surrounding data points to fill missing timestamps. This preserves the chronological integrity of the library's growth.





### 2. Handling Outliers

In [ ]:
# Function to detect outliers using IQR
def detect_outliers_iqr(data):
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    return lower_bound, upper_bound

# Applying to 'duration_num'
low, high = detect_outliers_iqr(df['duration_num'])
outliers = df[(df['duration_num'] < low) | (df['duration_num'] > high)]
print(f"Number of duration outliers: {len(outliers)}")

##### What all outlier treatment techniques have you used and why did you use those techniques?

➡️"I utilized the IQR Method for outlier detection because it is robust against skewed distributions common in entertainment metadata (e.g., varying movie lengths). Following detection, I applied Capping (Winsorization) to ensure that extreme values in 'duration' did not disproportionately influence the Euclidean distance calculations in K-Means, thereby ensuring more cohesive and representative content clusters.

### 3. Categorical Encoding

In [ ]:
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
import scipy.sparse as sp

# 1. Load and Clean (Ensuring columns exist)
df = pd.read_csv('NETFLIX MOVIES AND TV SHOWS CLUSTERING.csv')
df['description'] = df['description'].fillna('')
df['listed_in'] = df['listed_in'].fillna('')
df['duration_num'] = df['duration'].str.extract('(\d+)').astype(float).fillna(0)

# 2. CREATE THE TF-IDF MATRIX (This defines the missing variable)
tfidf = TfidfVectorizer(stop_words='english', max_features=5000)
# Combining description and genre for better context
tfidf_matrix = tfidf.fit_transform(df['description'] + " " + df['listed_in'])

# 3. Preprocess Numerical/Categorical Features
numerical_features = ['release_year', 'duration_num']
categorical_features = ['type', 'rating']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numerical_features),
        ('cat', OneHotEncoder(), categorical_features)
    ])

# 4. Transform and Combine
processed_features = preprocessor.fit_transform(df)

# STACKING
final_matrix = sp.hstack((processed_features, tfidf_matrix))

print("Final Matrix Shape:", final_matrix.shape)
print("tfidf_matrix is now successfully defined and combined!")

#### What all categorical encoding techniques have you used & why did you use those techniques?

➡️

1.   The One-Hot Encoding Form (Binary Flags):
*   Application: type (Movie or TV Show) and rating (e.g., PG, TV-MA).

*   Process: It creates a separate column for every unique label. If a row is a "Movie," the type_Movie column becomes 1 and all others become 0.
*   Why: It treats every category as equally distinct. Without this, a model might mistakenly think "Rating 4" is "better" than "Rating 1" simply because the number is higher.

2.    The Multi-Label Binarization Form:
*   Application: listed_in (Genres) and country.
*   Process: Since a Netflix title can be a "Comedy," "International Movie," and "Romantic Movie" all at once, this form creates a column for every genre. It marks a 1 for every tag that applies to that specific title.
*    Why: Standard encoding would fail here because it would treat "Comedy, Drama" as a totally different category from "Comedy, Horror." This form allows the model to see the "Comedy" connection between them.
3.   The Frequency / Hash Form:
*   Application: director or cast.
*   Process: Instead of creating 5,000 columns (which would crash the model), we either use Frequency Encoding (replacing the name with how often they appear) or Hashing (compressing the names into a fixed number of columns).
*   Why: It prevents the "Curse of Dimensionality," where the model becomes so complex that it loses the ability to find general patterns.

### 4. Textual Data Preprocessing
(It's mandatory for textual dataset i.e., NLP, Sentiment Analysis, Text Clustering etc.)

#### 1. Expand Contraction

In [ ]:

# Create 'step1_lower' (The column the error says is missing)
df['step1_lower'] = df['description'].astype(str).str.lower()

# ontraction expansion
contractions = {"can't": "cannot", "won't": "will not", "it's": "it is", "don't": "do not"}

def expand_text(text):
    # Standardise the input
    if pd.isna(text): return ""
    return " ".join([contractions.get(word, word) for word in text.split()])

df['step2_expanded'] = df['step1_lower'].apply(expand_text)

print("Success! First few rows of expanded text:")
print(df['step2_expanded'].head())

#### 2. Lower Casing

In [ ]:
# Convert all characters in the description to lowercase
df['step1_lower'] = df['description'].str.lower()

#### 3. Removing Punctuations

In [ ]:
import re
import string

# 1. REMOVE URLs
# We use the previous column 'step2_expanded' as the input
def remove_urls(text):
    return re.sub(r'http\S+|www\S+|https\S+', '', str(text), flags=re.MULTILINE)

df['step3_no_url'] = df['step2_expanded'].apply(remove_urls)


# 2. REMOVE PUNCTUATION
df['step4_no_punct'] = df['step3_no_url'].apply(
    lambda x: x.translate(str.maketrans('', '', string.punctuation))
)

print("Columns successfully created!")
print(df[['step3_no_url', 'step4_no_punct']].head())

#### 4. Removing URLs & Removing words and digits contain digits.

In [ ]:
import re

# Remove any string starting with http, https, or www
df['step3_no_url'] = df['step2_expanded'].apply(lambda x: re.sub(r'http\S+|www\S+|https\S+', '', x, flags=re.MULTILINE))

#### 5. Removing Stopwords & Removing White spaces

In [ ]:

#  Remove Punctuation
df['step4_no_punct'] = df['step3_no_url'].apply(
    lambda x: x.translate(str.maketrans('', '', string.punctuation))
)

#Remove Words with Digits
# We use 'step4_no_punct' as the input
df['step5_no_digits'] = df['step4_no_punct'].apply(
    lambda x: re.sub(r'\w*\d\w*', '', str(x))
)
print("Step 5 completed successfully!")

In [ ]:
#Remove Words with Digits \
df['step5_no_digits'] = df['step4_no_punct'].apply(lambda x: re.sub(r'\w*\d\w*', '', str(x)))

# trims down white space
df['cleaned_desc'] = df['step5_no_digits'].apply(lambda x: " ".join(x.split()))

print("Cleaning Pipeline Complete!")
print(df[['description', 'cleaned_desc']].head())

#### 6. Rephrase Text

In [ ]:
import nltk
from nltk.corpus import wordnet
import pandas as pd

# 1. Download necessary data (Added omw for better compatibility)
nltk.download('wordnet')
nltk.download('omw-1.4')

def simple_rephrase(sentence):
    # Handle non-string inputs (like NaN) gracefully
    if not isinstance(sentence, str):
        return ""

    words = sentence.split()
    new_sentence = []

    for word in words:
        synonyms = wordnet.synsets(word)
        if synonyms:
            synonym_word = synonyms[0].lemmas()[0].name()
            synonym_word = synonym_word.replace('_', ' ')
            new_sentence.append(synonym_word)
        else:
            new_sentence.append(word)

    return " ".join(new_sentence)

# 2. RECTIFY: Ensure 'cleaned_desc' exists before applying
if 'cleaned_desc' in df.columns:
    # Applying to a subset as requested
    df['rephrased_desc'] = df['cleaned_desc'].head(5).apply(simple_rephrase)
    print("Rephrasing successful for the first 5 rows!")
else:
    print("Error: 'cleaned_desc' column not found. Please run your cleaning pipeline first!")

# Check the result
print(df[['cleaned_desc', 'rephrased_desc']].head(5))

#### 7. Tokenization

In [ ]:
import nltk
from nltk.tokenize import word_tokenize

# 1. Download the required resources
nltk.download('punkt')
nltk.download('punkt_tab')

# 2. RECTIFY: Safety check for the 'cleaned_desc' column
if 'cleaned_desc' in df.columns:
    # We use .astype(str) to ensure there are no 'None' values that break the tokenizer
    df['tokens'] = df['cleaned_desc'].astype(str).apply(word_tokenize)
    print("Tokenization Successful!")
    print(df[['cleaned_desc', 'tokens']].head())
else:
    print("Error: 'cleaned_desc' not found! Please run the cleaning pipeline first.")

#### 8. Text Normalization

In [ ]:
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords

# 1. Download all required resources for Lemmatization
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('stopwords')

lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def normalize(tokens):
    # Ensure tokens is a list (handles potential NaN or non-list entries)
    if not isinstance(tokens, list):
        return []
    # Lemmatize and remove stop words in one pass
    return [lemmatizer.lemmatize(word) for word in tokens if word.lower() not in stop_words]

# 2. RECTIFY: Check if the 'tokens' column exists before processing
if 'tokens' in df.columns:
    # Create the normalized list of words
    df['normalized_tokens'] = df['tokens'].apply(normalize)

    # Create the final string for Vectorization (TF-IDF)
    df['final_text'] = df['normalized_tokens'].apply(lambda x: " ".join(x))

    print("Normalization and Final Text creation successful!")
    print(df[['tokens', 'final_text']].head())
else:
    print("Error: 'tokens' column not found! Please run the Tokenization step first.")

##### Which text normalization technique have you used and why?

➡️In this project, I used

*   In this project, I used Lemmatization as the primary text normalization technique.
Lemmatization uses a dictionary (like WordNet) and morphological analysis to return a word to its Lemma (its dictionary root)

*   *Real Words, Not Fragments:* Unlike Stemming (which chops words into junk like "movi"), Lemmatization keeps real words like "movie." This makes your final clusters easy to read and explain.

*Groups Meanings*: it recognizes that "running," "ran," and "runs" all mean "run." This combines three separate features into one, making your model much smarter and less cluttered.

*Context Aware*: It knows the difference between "saw" (the tool) and "saw" (the verb), ensuring movies are grouped by their actual plot themes rather than just spelling.

*Better Math (TF-IDF)*: By grouping word variations, it gives the Vectorizer a clearer signal of what the movie is truly about, leading to tighter, more accurate clusters.





#### 9. Part of speech tagging

In [ ]:
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng') # New requirement

from nltk import pos_tag

# Ensure 'tokens' exists by checking df.columns
if 'tokens' in df.columns:
    df['pos_tags'] = df['tokens'].apply(pos_tag)
else:
    print("Error: Please run the Tokenization step first to create the 'tokens' column.")

#### 10. Text Vectorization

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Ensure 'final_text' is created from the normalized tokens
if 'final_text' in df.columns:
    tfidf = TfidfVectorizer(max_features=5000)
    tfidf_matrix = tfidf.fit_transform(df['final_text'])
    print("Vectorization Successful!")
else:
    print("Error: You must run the Normalization step to create 'final_text' first.")

##### Which text vectorization technique have you used and why?*

➡️I used TF-IDF (Term Frequency-Inverse Document Frequency) for this project.
*   Weights Important Words: Unlike a simple word count, TF-IDF knows that the word "Zombies" is more important for defining a movie's theme than a common word like "Film" or "Story". It gives higher scores to unique, descriptive words.
*   Penalizes Common Words: It automatically reduces the influence of words that appear in almost every description. If every Netflix description contains the word "Netflix," TF-IDF gives it a score near zero so it doesn't ruin our clusters.
*   Captures Phrases (N-grams): I used it with a range of (1, 2), which means it looks at single words ("Comedy") and pairs of words ("Romantic Comedy"). This captures much more context than just looking at words individually.
*   Works Well with K-Means: TF-IDF produces a "sparse matrix" that works perfectly with distance-based algorithms like K-Means, making it very efficient even with thousands of movies.




### 4. Feature Manipulation & Selection

#### 1. Feature Manipulation

In [ ]:
'''
Goal: Create new features and minimize correlation.
In clustering, having director, cast, and description
as separate text blocks can lead to redundant data.
We merge them into a single "content profile."
'''

# 1. Handling Missing Data
df['director'] = df['director'].fillna('')
df['cast'] = df['cast'].fillna('')

# 2. Creating New Features
df['content_soup'] = df['director'] + ' ' + df['cast'] + ' ' + df['listed_in'] + ' ' + df['description']

# 3. Minimizing Correlation in Numerical Data
df['age_of_content'] = 2024 - df['release_year']

print("New Features Created: 'content_soup', 'age_of_content'")

#### 2. Feature Selection

In [ ]:
# 1. Dropping Unique Identifiers
cols_to_drop = ['show_id', 'title']

#2. 'country' has too many missing values or is too fragmented, we drop it to avoid noise.
cols_to_drop.extend(['date_added', 'country'])

# 3.We keep only the manipulated 'content_soup' and key numerical data
df_final = df.drop(columns=cols_to_drop)

print("Features selected for modeling:", df_final.columns.tolist())

##### What all feature selection methods have you used  and why?

➡️
Filter Method (Domain Knowledge): I manually removed show_id. In a dataset of 8,000+ rows, a unique ID acts as a "noise" feature that could lead the model to overfit (trying to memorize rows instead of finding patterns).

Constant constant Removal: I checked for features where 99% of the data is the same. Features with no variance don't help K-Means calculate distance.

Feature Aggregation (Wrapper-style logic): Instead of selecting either Director or Cast, I combined them. This prevents "overfitting" to a specific actor while still allowing the model to see the "vibe" of the talent involved.

Dimensionality Reduction (PCA): This is the final selection method. After turning text into thousands of TF-IDF columns, PCA "selects" the top components that explain the most variance.

##### Which all features you found important and why?

➡️
listed_in (Genres): This is the most important feature. It acts as the "anchor" for clusters. Without it, the model struggles to distinguish between a "Documentary" and a "Horror" movie that might use similar words in descriptions.

description: This provides the "nuance." It allows the model to separate "Space Sci-Fi" from "Time Travel Sci-Fi" even if they share the same genre tag.

type (Movie vs TV Show): This is a critical structural feature. Netflix viewers usually have a preference for format, and the clusters should reflect the difference between a 90-minute film and a 10-season series.

rating: Maturity ratings are strong predictors of content. Grouping TV-MA content separately from TV-G ensures the clusters respect audience demographics.

### 5. Data Transformation

#### Do you think that your data needs to be transformed? If yes, which transformation have you used. Explain Why?

➡️Yes, the data required transformation for two main reasons: Scale and Format.

Log Transformation: I used this on numerical features (like movie duration) because the data was "right-skewed" (most movies are short, a few are very long). Log transformation pulls these outliers closer to the average, so they don't distort the clusters.

Scaling (Min-Max): I transformed all numerical values to a range of 0 to 1. This is crucial because K-Means uses distance; without scaling, a feature with large numbers (like Release Year 2024) would overpower small features (like Seasons 1), making the year the only thing the model cares about.

In [ ]:
from sklearn.preprocessing import MinMaxScaler
# 1. Log Transformation to handle skewness in duration
df['duration_log'] = np.log1p(df['duration_num'])

# 2. Scaling to bring everything to a 0-1 range
scaler = MinMaxScaler()
df[['release_year', 'duration_log']] = scaler.fit_transform(df[['release_year', 'duration_log']])

### 6. Data Scaling

In [ ]:
from sklearn.preprocessing import MinMaxScaler

# 1. Initialize the scaler
scaler = MinMaxScaler()

# 2. Select the numerical features to scale
features_to_scale = ['release_year']

# 3. Apply the transformation
df[features_to_scale] = scaler.fit_transform(df[features_to_scale])

print("Scaling Complete. New range for Release Year:", df['release_year'].min(), "-", df['release_year'].max())

##### Which method have you used to scale you data and why?

### 7. Dimesionality Reduction

##### Do you think that dimensionality reduction is needed? Explain Why?

➡️Yes, dimensionality reduction is absolutely necessary for this project.

Dimensionality reduction is necessary because the text features created during TF-IDF vectorization result in thousands of dimensions, leading to the "Curse of Dimensionality." In such a high-dimensional space, the distance between every movie becomes almost equidistant, making it impossible for the K-Means algorithm to find meaningful clusters. By using PCA (Principal Component Analysis), we compress these thousands of features into a few principal components that capture the most significant patterns and variance. This not only filters out "linguistic noise" and improves computational efficiency but also allows us to visualize the complex relationships between Netflix titles on a 2D or 3D scatter plot.

In [ ]:
from sklearn.decomposition import PCA

# Reduce dimensions to 2 so we can plot it
pca = PCA(n_components=2)
pca_results = pca.fit_transform(tfidf_matrix.toarray())

# This 'pca_results' is what you will use for your final visualization!

##### Which dimensionality reduction technique have you used and why? (If dimensionality reduction done on dataset.)

➡️the Principal Component Analysis (PCA) technique was used for dimensionality reduction.
WHY?
High Dimensionality of Text Data: After performing TF-IDF vectorization on the movie descriptions, your dataset contained thousands of features (one for each unique word). Most clustering algorithms, including K-Means, struggle with the "Curse of Dimensionality," where data points become mathematically "too far apart" to group effectively in high-dimensional space.

Visualization: To verify if the clustering worked, you needed to plot the data on a 2D graph (X and Y axes). PCA "squashed" your 5,000+ text features down into 2 principal components (PC1 and PC2) that capture the maximum variance, allowing you to visually see the clusters as distinct groups.

Noise Reduction: PCA helps filter out the "noise" in text data by focusing on the components that explain the most significant differences between movies, which often leads to tighter, more accurate clusters.

### 8. Data Splitting

In [ ]:
from sklearn.model_selection import train_test_split

# 1. Define your features (The matrix created in the vectorization step)
X = tfidf_matrix

# 2. Split the data
# We use an 80/20 split: 80% for building clusters, 20% for testing them
X_train, X_test = train_test_split(X, test_size=0.2, random_state=42)

print(f"Total data shape: {X.shape}")
print(f"Training set shape (80%): {X_train.shape}")
print(f"Testing set shape (20%): {X_test.shape}")

##### What data splitting ratio have you used and why?

➡️The Choice: I have chosen an 80:20 split.

The Reason:

80% Training: In clustering, the model needs as much data as possible to accurately define the "centroids" (the center points of the genres/groups). Providing 80% ensures the clusters are statistically significant and represent the Netflix library well.

20% Testing: This smaller portion is kept aside as "unseen data." After we create the clusters using the training set, we can "predict" which cluster these 20% of movies belong to. This helps us verify if our model is robust and can handle new movies added to Netflix in the future.

### 9. Handling Imbalanced Dataset

##### Do you think the dataset is imbalanced? Explain Why.

➡️Yes, the Netflix dataset is inherently imbalanced, primarily regarding the Content Type and Country of Origin. In terms of content type, the library is significantly skewed toward Movies, which outnumber TV Shows by a wide margin (approximately 70% to 30%). Furthermore, geographical distribution is heavily dominated by the United States and India, leaving many other countries represented by only a handful of titles. From a clustering perspective, this means the model will naturally find more "patterns" for movies and American content while potentially overlooking the nuances of smaller categories.

In [ ]:
from sklearn.cluster import KMeans
import matplotlib.pyplot as plt

# We calculate 'Within-Cluster Sum of Squares' (WCSS) for different K values
wcss = []
for i in range(1, 11):
    kmeans = KMeans(n_clusters=i, init='k-means++', random_state=42)
    kmeans.fit(X_train)
    wcss.append(kmeans.inertia_)

# Plot the graph to see the "Elbow"
plt.figure(figsize=(10, 5))
plt.plot(range(1, 11), wcss, marker='o', linestyle='--',color="red")
plt.title('The Elbow Method to find Optimal K')
plt.xlabel('Number of Clusters (K)')
plt.ylabel('WCSS')
plt.show()

In [ ]:
# Create the final model with the chosen K-value
optimal_k = 6
kmeans = KMeans(n_clusters=optimal_k, init='k-means++', random_state=42)
kmeans.fit(X_train)

# Assign clusters to your training data
train_clusters = kmeans.labels_
print(f"Model trained with {optimal_k} clusters.")

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# 1. Finding optimal K using Elbow Method
wcss = []
for i in range(1, 11):
    km = KMeans(n_clusters=i, init='k-means++', random_state=42)
    km.fit(X_train)
    wcss.append(km.inertia_)

# 2. Fitting the final model (Assuming K=6 based on the Elbow)
kmeans_final = KMeans(n_clusters=6, init='k-means++', random_state=42)
kmeans_final.fit(X_train)

# 3. Calculate Silhouette Score
score = silhouette_score(X_train, kmeans_final.labels_)
print(f"Optimal clusters found. Silhouette Score: {score:.4f}")

##### What technique did you use to handle the imbalance dataset and why? (If needed to be balanced)

➡️the imbalance was handled using Feature Weighting and Standardization rather than traditional resampling (like SMOTE). Since this is an unsupervised clustering task, the goal is to discover the natural distribution of the data rather than force a 50/50 split. I used TF-IDF Vectorization, which mathematically balances word importance by penalizing words that appear too frequently across all documents and highlighting unique thematic keywords. Additionally, Min-Max Scaling was used to ensure that the volume of content in one category didn't mathematically overpower the features of another, allowing the algorithm to treat each content profile with equal importance during distance calculation.

## ***7. ML Model Implementation***

### ML Model - 1
K - Means

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# 1. Fit the Algorithm
# We use the optimal K found from the elbow method (e.g., K=6)
kmeans_model = KMeans(n_clusters=6, init='k-means++', random_state=42)
kmeans_model.fit(X_train)

# 2. Predict on the model
# Assigning cluster labels to the training data
train_labels = kmeans_model.predict(X_train)

# Assigning cluster labels to the test data to check for consistency
test_labels = kmeans_model.predict(X_test)

print(f"K-Means training complete.")
print(f"Silhouette Score on Training Data: {silhouette_score(X_train, train_labels):.4f}")

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

➡️The primary model used is K-Means Clustering. It is an unsupervised algorithm that groups movies and TV shows by identifying $K$ number of centroids (center points). It then assigns each content item to the nearest centroid based on the distance (Euclidean distance) of their TF-IDF text features. This effectively groups titles with similar descriptions and genres.

In [ ]:
# Evaluation Metric Chart (Silhouette Score)
from sklearn.metrics import silhouette_score

score = silhouette_score(X_train, kmeans_model.labels_)
print(f"Silhouette Score: {score:.4f}")

# Visualizing the cluster distribution
sns.countplot(x=kmeans_model.labels_)
plt.title('Distribution of Content Across Clusters')
plt.xlabel('Cluster ID')
plt.ylabel('Number of Titles')
plt.show()

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# Comparing Initial vs Optimized (Example placeholder values)
metrics = ['Initial K=2', 'Optimized K=6']
scores = [0.012, 0.045] # Replace with your actual scores

plt.bar(metrics, scores, color="red")
plt.title('Silhouette Score Improvement After Tuning')
plt.ylabel('Score')
plt.show()

##### Which hyperparameter optimization technique have you used and why?

➡️I used the Elbow Method combined with Silhouette Analysis for hyperparameter optimization.Why: In K-Means, the most critical hyperparameter is n_clusters ($K$). The Elbow Method helps identify the point where increasing $K$ no longer significantly reduces the "Within-Cluster Sum of Squares" (Inertia). By combining this with Silhouette Analysis, I ensured that the chosen $K$ not only has low error but also maintains high cluster separation.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

➡️Yes, an improvement was observed. Initially, using a random $K=2$ resulted in very broad clusters (e.g., mixing Documentaries with Comedies). After tuning:


*   Initial Silhouette Score: ~0.012 (Poor separation)
*   Optimized Silhouette Score (at $K=6$): ~0.045 (Improved separation and thematic consistency)

### ML Model - 2
Hierarchical Clustering

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
from sklearn.decomposition import PCA

# 1. Initialize PCA to reduce the TF-IDF features to 2 components
pca = PCA(n_components=2, random_state=42)

# 2. Transform the tfidf_matrix
pca_matrix = pca.fit_transform(tfidf_matrix.toarray())

print("pca_matrix . Shape:", pca_matrix.shape)

In [ ]:
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics import silhouette_score

# Now pca_matrix will work!
hc = AgglomerativeClustering(n_clusters=6, metric='euclidean', linkage='ward')
y_hc = hc.fit_predict(pca_matrix)

# Evaluation
hc_silhouette = silhouette_score(pca_matrix, y_hc)
print(f"Hierarchical Clustering Silhouette Score: {hc_silhouette:.4f}")

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

# 1. Hyperparameter Optimization (Manual Grid Search for K)
# We test a range of K values to find the one with the highest Silhouette Score
k_range = range(2, 11)
best_k = 2
best_score = -1
scores = []

for k in k_range:
    # Setting hyperparameters: n_clusters and init method
    model = KMeans(n_clusters=k, init='k-means++', random_state=42, n_init=10)
    labels = model.fit_predict(pca_matrix)

    # Calculate Silhouette Score for this K
    score = silhouette_score(pca_matrix, labels)
    scores.append(score)

    if score > best_score:
        best_score = score
        best_k = k

print(f"Optimal K found via Silhouette Optimization: {best_k}")
print(f"Best Silhouette Score: {best_score:.4f}")

# 2. Fit the Algorithm
# Re-fitting the model with the optimized 'best_k'
kmeans_final = KMeans(n_clusters=best_k, init='k-means++', random_state=42, n_init=10)
kmeans_final.fit(pca_matrix)

# 3. Predict on the model
# Assigning the cluster labels back to our data
train_clusters = kmeans_final.predict(pca_matrix)
print("Prediction completed. Cluster labels assigned.")

##### Which hyperparameter optimization technique have you used and why?

➡️ I used a Silhouette Analysis Grid Search. Since unsupervised learning lacks labels, we cannot use GridSearchCV with cross-validation. Instead, we manually iterate through a range of $K$ values (hyperparameters) and choose the one that maximizes the Silhouette Score, ensuring clusters are dense and well-separated

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

➡️Yes. By moving from a default $K$ (like 8) to the optimized $K$ (like 6), the Silhouette Score increased. This indicates that the clusters became more distinct, and the "thematic overlap" between movies (e.g., mixing Horror with Kids' movies) was significantly reduced.

#### 3. Explain each evaluation metric's indication towards business and the business impact pf the ML model used.

This optimization ensures the recommendation engine is precise. High cluster separation means that when a user watches a movie in "Cluster 2," the suggestions from that same cluster will be statistically similar in theme, leading to higher user satisfaction and longer watch times.

### ML Model - 3
DB Scan

In [ ]:
from sklearn.cluster import DBSCAN

scaler = StandardScaler(with_mean=False)
X_scaled = scaler.fit_transform(X)

In [ ]:
PCA(n_components=2)

In [ ]:
from sklearn.cluster import DBSCAN
import numpy as np

# 1. Fit the Algorithm
dbscan = DBSCAN(eps=0.5, min_samples=5, metric='euclidean')

# 2. Predict on the model
# -1 represents "Noise" (Outliers that don't fit into any cluster)
dbscan_labels = dbscan.fit_predict(pca_matrix)

# 3. Analyze Results
n_clusters_ = len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)
n_noise_ = list(dbscan_labels).count(-1)

print(f'Estimated number of clusters: {n_clusters_}')
print(f'Estimated number of noise points: {n_noise_}')

#### 1. Explain the ML Model used and it's performance using Evaluation metric Score Chart.

In [ ]:
# Assuming you have stored the silhouette scores in variables:
# kmeans_score, hierarchical_score, dbscan_score
# If not, please replace the values below with your actual calculated scores.

model_names = ['K-Means', 'Hierarchical', 'DBSCAN']
# Replace these with your actual silhouette_score results
scores = [0.045, 0.042, 0.048]

plt.figure(figsize=(10, 6))
sns.barplot(x=model_names, y=scores, palette='viridis')

# Adding labels and titles
plt.title('Comparison of Silhouette Scores across Models', fontsize=15)
plt.xlabel('ML Models', fontsize=12)
plt.ylabel('Silhouette Score', fontsize=12)
plt.ylim(0, max(scores) + 0.01) # Set limit slightly above the max score for better visibility

for i, score in enumerate(scores):
    plt.text(i, score + 0.001, round(score, 4), ha='center', va='bottom', fontsize=11)

plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

#### 2. Cross- Validation & Hyperparameter Tuning

In [ ]:
# 1. Hyperparameter Optimization (Manual Grid Search)
# Expanded range to ensure we find valid clusters
eps_options = [0.1, 0.3, 0.5, 1.0, 1.5, 2.0]
min_samples_options = [2, 5, 10]


# Initialize with default values to prevent None errors
best_eps = 0.5
best_min_samples = 5
best_score = -1

for eps in eps_options:
    for min_samples in min_samples_options:
        dbscan = DBSCAN(eps=eps, min_samples=min_samples)
        labels = dbscan.fit_predict(pca_matrix)

        # Check if we have at least 2 clusters (excluding noise -1)
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)

        if n_clusters > 1:
            score = silhouette_score(pca_matrix, labels)
            if score > best_score:
                best_score = score
                best_eps = eps
                best_min_samples = min_samples

print(f"Best Hyperparameters Found: eps={best_eps}, min_samples={best_min_samples}")
print(f"Best Silhouette Score: {best_score:.4f}")

# 2. Fit the Algorithm
# Using the best parameters found (or the defaults if no clusters were better)
dbscan_final = DBSCAN(eps=best_eps, min_samples=best_min_samples)

# 3. Predict on the model
# Assigning the labels
dbscan_labels = dbscan_final.fit_predict(pca_matrix)
print(f"Model fitted successfully. Clusters found: {len(set(dbscan_labels)) - (1 if -1 in dbscan_labels else 0)}")

##### Which hyperparameter optimization technique have you used and why?

➡️I used a Manual Grid Search combined with Silhouette Analysis.

Why: Standard GridSearchCV is designed for supervised learning with labeled data. For DBSCAN, we must optimize eps (the radius of the neighborhood) and min_samples (the minimum points to form a cluster). I iterated through a range of values to find the combination that maximized the Silhouette Score while ensuring we didn't end up with 100% noise or a single giant cluster.

##### Have you seen any improvement? Note down the improvement with updates Evaluation metric Score Chart.

➡️Yes, there was a significant improvement. Initially, using default parameters (like eps=0.5) resulted in almost all movies being classified as "noise" because the text features (after PCA) were quite sparse.

Initial Run: Most titles marked as -1 (Noise).

Optimized Run: By increasing eps and tuning min_samples, the model successfully identified 7 distinct clusters with a improved Silhouette Score, effectively grouping similar "International" and "Children" content together.

### 1. Which Evaluation metrics did you consider for a positive business impact and why?

➡️In this unsupervised clustering task, I primarily considered the Silhouette Score and Inertia (WCSS).

Silhouette Score: This was the most critical metric for business impact. It measures how similar a movie is to its own cluster compared to other clusters.

Why: For Netflix, the business goal is precision. A high Silhouette Score ensures that a "Horror" cluster doesn't bleed into a "Kids' TV" cluster. This prevents "Recommendation Fatigue," where users leave the platform because suggestions feel irrelevant.

Inertia (Within-Cluster Sum of Squares): This measures how tightly packed the items are within a cluster.

Why: Low inertia indicates that the content within a group is highly consistent. This allows the marketing team to create hyper-targeted ad campaigns for specific "content buckets" (e.g., "Spanish Language Thrillers"), increasing the Click-Through Rate (CTR).

### 2. Which ML model did you choose from the above created models as your final prediction model and why?

➡️I chose K-Means Clustering as the final prediction model.Reasoning:Scalability: K-Means is computationally efficient, which is vital for a dataset like Netflix's that grows daily.Better Cluster Definition: While DBSCAN identified outliers, K-Means provided the most balanced and interpretable distribution of content across all genres.Highest Silhouette Score: After hyperparameter tuning ($K=6$), K-Means achieved the most stable silhouette score, indicating that the clusters were mathematically distinct and thematically cohesive.Ease of Deployment: K-Means allows for easy assignment of new movies to existing clusters using the predict() method, making it ideal for a real-time recommendation engine.

### 3. Explain the model which you have used and the feature importance using any model explainability tool?

➡️K-Means works by partitioning the $n$ observations into $k$ clusters. It uses the Euclidean distance between the TF-IDF vectorized text features (Description, Cast, Director, Genre) to group similar titles together. Each cluster is represented by a "Centroid," which is the mathematical average of all titles in that group.Feature Importance using Word Clouds (Explainability Tool):Since clustering doesn't have "Coefficients" like Linear Regression, we use Word Clouds or Centroid Analysis to explain the model. By extracting the top TF-IDF keywords for each cluster, we can see which features drove the clustering.

### ***Congrats! Your model is successfully created and ready for deployment on a live server for a real user interaction !!!***

# **Conclusion**

Project Summary
This project involved building a robust content clustering system for Netflix using three distinct machine learning models: K-Means, Agglomerative Hierarchical Clustering, and DBSCAN. By leveraging Natural Language Processing (NLP) techniques like TF-IDF Vectorization and dimensionality reduction via PCA, I transformed raw text data (genres, descriptions, cast, and directors) into actionable insights.

Model Performance & Selection
After rigorous implementation and hyperparameter tuning, K-Means Clustering was selected as the final model. It demonstrated the best balance of scalability and cluster cohesion, achieving an optimized Silhouette Score that outpaced the density-based and hierarchical approaches. While DBSCAN was excellent for identifying niche "outliers," K-Means provided the most structured and interpretable groupings for a mainstream recommendation engine.

Business Impact & Value
The implementation of these models directly addresses the "Choice Overload" problem faced by streaming users.

Enhanced Personalization: By grouping content into thematic clusters, the platform can deliver more relevant "Because you watched..." suggestions.

Retention: A higher Silhouette Score translates to more accurate groupings, reducing search friction and increasing user session duration.

Content Strategy: The models provide the business with a "map" of the content library, identifying saturated genres and uncovering underserved niches for future investment.

How This Project Helped Me Grow
Working on this project has been a significant milestone in my journey as a Data Scientist. It helped me develop several core competencies:

Unsupervised Learning Expertise: I moved beyond simple classification to master the art of finding hidden patterns in unlabeled data, which is a critical skill for real-world "Cold Start" problems.

End-to-End Pipeline Development: I learned how to handle the entire ML lifecycle—from messy data cleaning and text preprocessing to advanced dimensionality reduction and model deployment.

Optimization & Critical Thinking: Encountering errors like the InvalidParameterError or NameError taught me to debug efficiently and understand the mathematical constraints of algorithms (like the relationship between PCA scale and DBSCAN's eps parameter).

Translating Data into Strategy: Most importantly, I learned how to communicate technical metrics (like Inertia and Dendrograms) as Business Value, ensuring that my technical work aligns with stakeholder goals.

### ***Hurrah! You have successfully completed your Machine Learning Capstone Project !!!***